# Point of this Notebook - Lakebase OLTP Layer

## What we did
- Provisioned **Lakebase** (managed Postgres) as the project's operational DB.
- Created 3 tables with **enforced primary keys**: `portfolio_positions`, `trade_log`, `alerts`.
- Seeded them from `gold.daily_signals` using **idempotent upserts** — re-runs never duplicate.
- Demonstrated **OLTP ops**: point lookup, single-row update (acknowledge alert), portfolio rollup.
## Why it's needed
- Delta `gold.*` = **analytics (OLAP)** — built for big scans, slow for tiny single-row writes.
- The app needs **live state** it can read/update one row at a time, fast, with integrity.
- That's an **OLTP** job → Lakebase (row-store Postgres + real PK indexes) is the right tool.
- Result: clean **OLAP + OLTP split** — Delta/Spark for data+ML, Lakebase for app state.

In [0]:
%pip install --quiet "psycopg[binary]" "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

w.secrets.put_secret("my_scope","my_postgres_secret",string_value ="<TOKEN>")

Lakebase - https://cust-e2-us-west-2.cloud.databricks.com/lakebase/projects/e0f7c799-935d-4919-ac53-a4fe3818737d?database=databricks_postgres

In [0]:
import psycopg
CONN_STR = "postgresql://raghavan.vaidhyaraman%40databricks.com@ep-red-frog-d128t6zg.database.us-west-2.cloud.databricks.com/databricks_postgres?sslmode=require"
OAUTH_TOKEN = dbutils.secrets.get(scope="my_scope", key="my_postgres_secret")
conn = psycopg.connect(CONN_STR, password=OAUTH_TOKEN)
conn.autocommit = True
PG_USER = "raghavan.vaidhyaraman@databricks.com"

### Tables

In [0]:
DDL = """
CREATE TABLE IF NOT EXISTS portfolio_positions (
  position_id     UUID PRIMARY KEY DEFAULT gen_random_uuid(),
  symbol          TEXT NOT NULL,
  quantity        DOUBLE PRECISION NOT NULL DEFAULT 0,
  avg_entry_price DOUBLE PRECISION,
  current_price   DOUBLE PRECISION,
  unrealized_pnl  DOUBLE PRECISION,
  position_type   TEXT DEFAULT 'LONG',
  opened_at       TIMESTAMP DEFAULT now(),
  updated_at      TIMESTAMP DEFAULT now(),
  is_open         BOOLEAN DEFAULT TRUE,
  CONSTRAINT uq_positions_symbol UNIQUE (symbol)
);

CREATE TABLE IF NOT EXISTS trade_log (
  trade_id        UUID PRIMARY KEY DEFAULT gen_random_uuid(),
  symbol          TEXT NOT NULL,
  trade_date      DATE NOT NULL,
  action          TEXT NOT NULL,
  quantity        DOUBLE PRECISION NOT NULL,
  price           DOUBLE PRECISION NOT NULL,
  signal_confidence DOUBLE PRECISION,
  signal_source   TEXT DEFAULT 'model_v1',
  executed_at     TIMESTAMP DEFAULT now(),
  notes           TEXT,
  CONSTRAINT uq_trades_symbol_day UNIQUE (symbol, trade_date)
);

CREATE TABLE IF NOT EXISTS alerts (
  alert_id        UUID PRIMARY KEY DEFAULT gen_random_uuid(),
  alert_type      TEXT NOT NULL,
  symbol          TEXT,
  message         TEXT NOT NULL,
  severity        TEXT DEFAULT 'INFO',
  is_acknowledged BOOLEAN DEFAULT FALSE,
  created_at      TIMESTAMP DEFAULT now(),
  acknowledged_at TIMESTAMP,
  acknowledged_by TEXT,
  CONSTRAINT uq_alerts_type_msg UNIQUE (alert_type, message)
);
"""
with conn.cursor() as cur:
    cur.execute(DDL)
print("Lakebase tables created.")

### Populate from today's signals (idempotent upsert)
Read the signals from the Gold Delta table with Spark, then insert into Postgres via psycopg.

In [0]:
from pyspark.sql.functions import col

CATALOG = "raghavan_trading_signals"
ds = spark.table(f"{CATALOG}.gold.daily_signals")
latest_date = ds.agg({"trade_date": "max"}).collect()[0][0]
rows = ds.filter(col("trade_date") == latest_date).collect()

with conn.cursor() as cur:
    # one trade per symbol per day
    cur.executemany(
        """INSERT INTO trade_log (symbol, trade_date, action, quantity, price, signal_confidence)
           VALUES (%s, %s, %s, %s, %s, %s)
           ON CONFLICT (symbol, trade_date)
           DO UPDATE SET action = EXCLUDED.action, price = EXCLUDED.price, executed_at = now();""",
        [(r["symbol"], r["trade_date"], r["signal"], 100.0, float(r["close_price"]), 0.55)
         for r in rows])

    # one open position per BUY symbol
    cur.executemany(
        """INSERT INTO portfolio_positions
             (symbol, quantity, avg_entry_price, current_price, unrealized_pnl, position_type, is_open)
           VALUES (%s, %s, %s, %s, 0, 'LONG', TRUE)
           ON CONFLICT (symbol)
           DO UPDATE SET current_price = EXCLUDED.current_price, updated_at = now();""",
        [(r["symbol"], 100.0, float(r["close_price"]), float(r["close_price"]))
         for r in rows if r["signal"] == "BUY"])

    # a single demo alert
    cur.execute(
        """INSERT INTO alerts (alert_type, symbol, message, severity)
           VALUES ('REGIME_CHANGE', NULL, 'Market regime shifted toward higher volatility.', 'WARNING')
           ON CONFLICT (alert_type, message) DO NOTHING;""")

print("Seeded Lakebase tables")

## Demonstrate OLTP point operations (psycopg)

In [0]:
with conn.cursor() as cur:
    # point lookup on an indexed column
    cur.execute("SELECT symbol, quantity, avg_entry_price FROM portfolio_positions "
                "WHERE symbol = 'AAPL' AND is_open = TRUE;")
    print("AAPL position:", cur.fetchall())

    # single-row update — exactly what the app's "acknowledge alert" button does
    cur.execute("UPDATE alerts SET is_acknowledged = TRUE, acknowledged_at = now(), "
                "acknowledged_by = %s WHERE alert_type = 'REGIME_CHANGE' AND is_acknowledged = FALSE;",
                (PG_USER,))
    print("Acknowledged alerts:", cur.rowcount)

    # portfolio rollup
    cur.execute("SELECT COUNT(*) AS open_positions, SUM(quantity*avg_entry_price) AS total_exposure "
                "FROM portfolio_positions WHERE is_open = TRUE;")
    print("Rollup:", cur.fetchone())

    #For checking count of all tables
    for t in ["portfolio_positions", "trade_log", "alerts"]:
        cur.execute(f"SELECT COUNT(*) FROM {t};")
        print(f"{t:20s}: {cur.fetchone()[0]} rows")

## App permission

In [0]:
APP_CLIENT_ID = "c8760585-53b6-414b-abef-7a07d276bf0b"

with conn.cursor() as cur:
    for s in [
        f'GRANT CONNECT ON DATABASE databricks_postgres TO "{APP_CLIENT_ID}";',
        f'GRANT USAGE ON SCHEMA public TO "{APP_CLIENT_ID}";',
        f'GRANT SELECT, INSERT, UPDATE ON ALL TABLES IN SCHEMA public TO "{APP_CLIENT_ID}";',
        f'ALTER DEFAULT PRIVILEGES IN SCHEMA public '
        f'GRANT SELECT, INSERT, UPDATE ON TABLES TO "{APP_CLIENT_ID}";',
    ]:
        cur.execute(s)
print("Granted.")